# Compare every OCR backend on the French receipts (Kaggle T4)

Runs `scripts/evaluate_all_backends.py`: Paddle, PP-OCRv4, hybrid CLIP+SmolLM2 VLM, from-scratch
OCR-VLM, Groq, and Moondream -> canonical Ticket, scored by the same metrics (incl. read_acc) on the
18 French photos. A backend whose dependency / checkpoint / key is missing is skipped, not fatal.

**Add Input (attach datasets):**
1. code bundle - `receipt_vlm_colab_bundle.zip`
2. `receipt-vlm-french-eval` - `images_tickets_caisse/` + `real_labels/`
3. `receipt-vlm-hybrid-ckpt` - `receipt_vlm_500m_merged.pt`
4. `receipt-vlm-eval-ckpts` - `ocr_vlm_epoch*.pt` + `tokenizer.json` (from-scratch)
5. (optional) Moondream `.mf` weights dataset

**Settings:** GPU T4, Internet ON. **Secrets:** add `GROQ_API_KEY` (Add-ons -> Secrets) for the Groq column.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# --- materialize the code bundle ---
import glob, os, shutil, zipfile
from pathlib import Path
WORK = Path("/kaggle/working/repo")
def materialize():
    zips = glob.glob("/kaggle/input/**/receipt_vlm_colab_bundle.zip", recursive=True)
    if zips:
        WORK.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(WORK)
        return
    hits = glob.glob("/kaggle/input/**/vlm_training/scripts/evaluate_all_backends.py", recursive=True)
    if hits:
        dest = WORK / "dev_ocr"
        if not dest.exists():
            shutil.copytree(Path(hits[0]).resolve().parents[2], dest)
        return
    raise FileNotFoundError("code bundle not found in /kaggle/input -- Add Input the rebuilt bundle")
if not list(WORK.glob("**/vlm_training/scripts/evaluate_all_backends.py")):
    materialize()
hits = glob.glob(str(WORK / "**/vlm_training/scripts/evaluate_all_backends.py"), recursive=True)
assert hits, "evaluate_all_backends.py not found -- rebuild the bundle (scripts/zip_selfcontained_colab.py) and re-upload"
TRAIN_PKG = Path(hits[0]).resolve().parents[1]
DEV_OCR = TRAIN_PKG.parent
os.chdir(TRAIN_PKG)
print("Train package:", TRAIN_PKG)

In [ ]:
# --- install deps for every backend ---
import subprocess, sys
def pip(*a):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("-r", "requirements-training.txt", "tokenizers>=0.22,<=0.23")   # torch/transformers (hybrid + from-scratch)
pip("paddleocr", "paddlepaddle")                                     # paddle + ppocrv4 (CPU wheel)
pip("groq")                                                          # groq baseline
pip("-e", str(DEV_OCR))     # receipt_ocr (paddle/groq/moondream backends + parser)
pip("-e", str(TRAIN_PKG))   # receipt_vlm (hybrid + from-scratch models)
print("Install OK")

In [ ]:
# --- locate attached datasets + Groq secret ---
import glob, os
def _find(pattern):
    hits = glob.glob(pattern, recursive=True)
    return sorted(hits)[-1] if hits else None
def _find_dir_with(child):
    for d in glob.glob("/kaggle/input/**/" + child, recursive=True):
        if os.path.isdir(d):
            return os.path.dirname(d)
    return None
# French test set: a dir holding real_labels/ (images_tickets_caisse/ sits beside it)
FR_BASE = _find_dir_with("real_labels")
FR_IMAGES = os.path.join(FR_BASE, "images_tickets_caisse") if FR_BASE else None
FR_LABELS = os.path.join(FR_BASE, "real_labels") if FR_BASE else None
HYBRID_CKPT = _find("/kaggle/input/**/receipt_vlm_500m_merged.pt")
OCRVLM_CKPT = _find("/kaggle/input/**/ocr_vlm_epoch*.pt")
OCRVLM_TOK  = _find("/kaggle/input/**/tokenizer.json")
MOON_DIR    = _find_dir_with("*.mf") or None   # optional; None if no weights attached
# Groq key from Kaggle Secrets -> env
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["GROQ_API_KEY"] = UserSecretsClient().get_secret("GROQ_API_KEY")
    print("Groq key: loaded")
except Exception as e:
    print("Groq key: not set (", e, ") -> groq column will skip")
print("FR_IMAGES  ->", FR_IMAGES)
print("FR_LABELS  ->", FR_LABELS)
print("HYBRID     ->", HYBRID_CKPT)
print("OCRVLM     ->", OCRVLM_CKPT)
print("OCRVLM_TOK ->", OCRVLM_TOK)
print("MOONDREAM  ->", MOON_DIR)
assert FR_IMAGES and FR_LABELS, "attach the receipt-vlm-french-eval dataset (images_tickets_caisse + real_labels)"

In [ ]:
# --- run the comparison ---
import subprocess, sys
cmd = [sys.executable, "-u", "scripts/evaluate_all_backends.py",
       "--backends", "paddle", "ppocrv4", "hybrid", "groq", "moondream", "ocrvlm",
       "--french-images", FR_IMAGES, "--french-labels", FR_LABELS,
       "--output", "/kaggle/working/backend_comparison.json"]
if HYBRID_CKPT: cmd += ["--hybrid-checkpoint", HYBRID_CKPT]
if OCRVLM_CKPT: cmd += ["--ocrvlm-checkpoint", OCRVLM_CKPT]
if OCRVLM_TOK:  cmd += ["--ocrvlm-tokenizer", OCRVLM_TOK]
if MOON_DIR:    cmd += ["--moondream-weights", MOON_DIR]
print(">>", " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)